# Infographix: Neural Poster Layout Generator — Diagnostic & Full Training Pipeline

This notebook trains and evaluates **`TemplateConditionedLayoutModel`** (`src/neural_layout/`) on **SciPostLayout v2** with **ClearML** experiment tracking and model registry integration.

### Root-Cause Fixes Included in Source Modules
1. **Canvas-Bounded Box Parameterization (`bbox_parameterization="bounded"`)**: Predicts `w, h` bounded above `[w_min, h_min]` and scales `x0, y0` by `(1 - w), (1 - h)`, guaranteeing `0 <= x0, y0` and `x0 + w <= 1, y0 + h <= 1` by construction.
2. **Anchor-Conditioned Template Decoder (`use_anchor_conditioning=True`)**: Computes category-aware soft attention over retrieved template slots to extract explicit geometric anchor boxes and learns bounded refinements rather than collapsing to the dataset mean box.
3. **Scale-Invariant Overlap & Anti-Collapse Losses (`MultiTaskLayoutLoss`)**: Replaces raw intersection-area-over-$M^2$ normalization with per-element summed pairwise IoU, removes the blanket exemption for all figure–caption pairs (exempting only genuine ground-truth overlapping pairs via `target_overlap_exempt_mask`), and adds log-dimension `loss_size` supervision.
4. **Leak-Free Permutation-Equivariant Inputs (`include_order_index=False`)**: Removes the arbitrary shuffled sequence index (`dim 17`), preserves stable `element_ids` across collation, guards against 0-element posters and `estimated_column_count == 6` out-of-range targets, and caches PDF metadata lookups.

## Section A — Environment and Configuration

In [ ]:
import json
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("Running in Google Colab environment.")
    # Install dependencies if needed (uncomment if running in a fresh Colab runtime)
    # !pip install -q clearml pyarrow pandas scikit-learn matplotlib scipy pyyaml highspy
    # !apt-get -qq install -y poppler-utils

    # Mount Google Drive if datasets/repo live on Drive (optional)
    # from google.colab import drive
    # drive.mount('/content/drive')

    try:
        from google.colab import userdata
        for key_name, default_val in [
            ("CLEARML_API_ACCESS_KEY", None),
            ("CLEARML_API_SECRET_KEY", None),
            ("CLEARML_API_HOST", "https://api.clear.ml"),
            ("CLEARML_WEB_HOST", "https://app.clear.ml"),
            ("CLEARML_FILES_HOST", "https://files.clear.ml"),
        ]:
            val = None
            try:
                val = userdata.get(key_name)
            except Exception:
                val = None
            if val:
                os.environ[key_name] = val
            elif default_val and key_name not in os.environ:
                os.environ[key_name] = default_val
        print("ClearML Colab Secrets check complete.")
    except Exception as exc:
        print(f"Colab userdata unavailable: {exc}")

# Resolve repository root portably (works from repo root, notebooks/, or Colab)
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(
        f"Could not locate 'src/' under {PROJECT_ROOT}. Set working directory to repository root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.adaptation.layout_adapter import LayoutAdapter
from src.features.request_schema import simulate_request_from_heldout_poster
from src.models.layout_retrieval import LayoutRetriever
from src.neural_layout.dataset import (
    CATEGORY_TO_IDX,
    KNOWN_CATEGORIES,
    PosterLayoutDataset,
    build_synthetic_request_and_sample,
    collate_layout_batch,
)
from src.neural_layout.evaluator import LayoutEvaluator
from src.neural_layout.layout_model import TemplateConditionedLayoutModel
from src.neural_layout.losses import LossWeights, MultiTaskLayoutLoss
from src.neural_layout.trainer import LayoutTrainer, TrainingConfig, set_seed
from src.neural_layout.visualize import (
    render_four_panel_comparison,
    render_training_curves,
    render_tri_layout_comparison,
)
from src.registry.clearml_registry import (
    compute_train_poster_ids_sha256,
    extract_model_config_from_checkpoint,
    extract_training_config_from_checkpoint,
    load_registered_retriever,
    register_model,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Project Root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM (GB)    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f}")
print(f"pdftotext    : {shutil.which('pdftotext')}")
print(f"pdfimages    : {shutil.which('pdfimages')}")

In [ ]:
# Centralized Training & Experiment Configuration
SEED = 42
set_seed(SEED)

# Auto-resolve feature directory (prefers v2 if present, falls back to v1)
FEATURES_DIR = next(
    (
        PROJECT_ROOT / "data" / "processed" / "features" / sub
        for sub in ("v2", "v1")
        if (PROJECT_ROOT / "data" / "processed" / "features" / sub / "posters.parquet").exists()
    ),
    None,
)
if FEATURES_DIR is None:
    raise FileNotFoundError("Missing posters.parquet under data/processed/features/{v2,v1}")

PDF_DIR = PROJECT_ROOT / "data" / "interim" / "scipostlayout" / "poster" / "pdf"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints" / "neural_layout_v2"
BASELINE_CKPT_PATH = PROJECT_ROOT / "models" / "checkpoints" / "neural_layout" / "best_model.pt"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "neural_layout_v2_eval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Explicit ClearML retriever selection (never implicit latest/production or local models/registry)
RETRIEVER_MODEL_NAME = "poster_layout_retriever"
RETRIEVER_VERSION = "2.1.0"  # Supply exact semantic version registered in ClearML
RETRIEVER_MODEL_ID = None    # Optional exact ClearML model_id (takes precedence if set)

# Feature extraction & ablation flags
USE_PDF_EXTRACTION = PDF_DIR.exists() and shutil.which("pdftotext") is not None
INCLUDE_ORDER_INDEX = False  # Set False for permutation-equivariant Set Transformer input
RUN_ORDER_INDEX_ABLATION = True
RUN_BASELINE_ARCH_ABLATION = False  # Optional: compare independent baseline vs bounded anchor
ENABLE_CLEARML = True
RESUME_FROM_CHECKPOINT = None  # e.g. CHECKPOINT_DIR / "latest_checkpoint.pt"

LOSS_WEIGHTS = LossWeights(
    w_box=1.5,
    w_size=0.5,
    w_giou=1.5,
    w_bound=2.5,
    w_overlap=1.5,
    w_col=0.8,
    w_aspect=1.0,
    w_order=0.8,
    w_macro=0.4,
)

EXEC_ENV = "colab_gpu" if (IN_COLAB and DEVICE == "cuda") else ("colab_cpu" if IN_COLAB else DEVICE)
TRAIN_CONFIG = TrainingConfig(
    seed=SEED,
    batch_size=16,
    learning_rate=3e-4,
    weight_decay=1e-4,
    num_epochs=30,
    grad_clip_norm=1.0,
    optimizer_name="adamw",
    scheduler_name="cosine_warmup",
    warmup_epochs=2,
    use_amp=(DEVICE == "cuda"),
    early_stopping_patience=8,
    checkpoint_freq=5,
    num_workers=2 if IN_COLAB else 0,
    d_model=128,
    num_heads=4,
    encoder_layers=3,
    decoder_layers=3,
    ffn_dim=256,
    dropout=0.1,
    bbox_parameterization="bounded",
    use_anchor_conditioning=True,
    min_box_width=0.01,
    min_box_height=0.005,
    include_order_index=INCLUDE_ORDER_INDEX,
    retriever_model_name=RETRIEVER_MODEL_NAME,
    retriever_model_id=RETRIEVER_MODEL_ID,
    retriever_version=RETRIEVER_VERSION,
    loss_weights=LOSS_WEIGHTS,
    checkpoint_dir=CHECKPOINT_DIR,
    project_name="Infographix",
    task_name=f"Neural Layout Anchor+Bounded Training ({EXEC_ENV.upper()})",
    tags=["neural_layout", "anchor_bounded", EXEC_ENV],
    log_to_clearml=ENABLE_CLEARML,
    execution_env=EXEC_ENV,
    dataset_version="SciPostLayout v2 (70/15/15)",
    device=DEVICE,
)

print(f"FEATURES_DIR   : {FEATURES_DIR}")
print(f"PDF_DIR        : {PDF_DIR} (exists={PDF_DIR.exists()}, extract={USE_PDF_EXTRACTION})")
print(f"CHECKPOINT_DIR : {CHECKPOINT_DIR}")
print(f"TRAIN_CONFIG   : {TRAIN_CONFIG}")

## Section B — Dataset Validation

Loads the full `train`, `dev`, and `test` splits, fits `LayoutRetriever` **strictly on `train`**, and verifies split disjointness, anti-leakage exclusions, box bounds, missingness flags, batch masks, and `element_ids` survival through collation.

In [ ]:
posters_df = pd.read_parquet(FEATURES_DIR / "posters.parquet")
elements_df = pd.read_parquet(FEATURES_DIR / "elements.parquet")

report_path = FEATURES_DIR / "feature_report.json"
if report_path.exists():
    feature_report = json.loads(report_path.read_text(encoding="utf-8"))
    print("Feature Report Split Policy:", feature_report.get("quality_summary", {}).get("split_policy"))

print("\nActual Poster Counts by Split:")
print(posters_df["split"].value_counts().to_frame("posters").join(
    posters_df[posters_df["total_element_count"] > 0]["split"].value_counts().to_frame("non_empty_posters")
))

# Verify strict split disjointness
train_pids = set(posters_df.loc[posters_df["split"] == "train", "poster_id"])
dev_pids = set(posters_df.loc[posters_df["split"] == "dev", "poster_id"])
test_pids = set(posters_df.loc[posters_df["split"] == "test", "poster_id"])
assert len(train_pids & dev_pids) == 0, "Leakage: train & dev poster IDs overlap!"
assert len(train_pids & test_pids) == 0, "Leakage: train & test poster IDs overlap!"
assert len(dev_pids & test_pids) == 0, "Leakage: dev & test poster IDs overlap!"

# Load LayoutRetriever deterministically from ClearML Model Registry (explicit ID or version)
retriever_model, retriever_manifest = load_registered_retriever(
    model_id=RETRIEVER_MODEL_ID,
    semantic_version=RETRIEVER_VERSION if RETRIEVER_MODEL_ID is None else None,
    return_manifest=True,
)
TRAIN_CONFIG.retriever_model_id = retriever_model.clearml_model_id
TRAIN_CONFIG.retriever_version = retriever_model.clearml_model_version

train_posters_only = posters_df[posters_df["split"] == "train"].copy().reset_index(drop=True)
valid_train_posters = train_posters_only[train_posters_only["known_element_count"].fillna(0) > 0]
expected_train_hash = compute_train_poster_ids_sha256(valid_train_posters)
loaded_train_hash = retriever_manifest.get("model_config", {}).get("train_poster_ids_sha256")

if loaded_train_hash != expected_train_hash:
    print(
        f"Loaded ClearML retriever (ID={retriever_model.clearml_model_id}, v{retriever_model.clearml_model_version}) "
        "was fitted on a different train split; re-fitting strictly on current train_posters_only."
    )
    retriever_model.fit(train_posters_only)
else:
    print(
        f"Verified ClearML retriever (ID={retriever_model.clearml_model_id}, v{retriever_model.clearml_model_version}) "
        "matches current train split exactly."
    )
print(f"Retriever ready on {len(retriever_model.train_posters_):,} train posters.")

# Pre-index elements once for O(1) lookup across all splits
elements_by_poster = {int(pid): grp for pid, grp in elements_df.groupby("poster_id", sort=False)}

train_dataset = PosterLayoutDataset(
    posters_df=posters_df,
    elements_df=elements_df,
    retriever_model=retriever_model,
    split="train",
    pdf_base_dir=PDF_DIR,
    include_order_index=INCLUDE_ORDER_INDEX,
    use_pdf_extraction=USE_PDF_EXTRACTION,
    elements_by_poster=elements_by_poster,
)
dev_dataset = PosterLayoutDataset(
    posters_df=posters_df,
    elements_df=elements_df,
    retriever_model=retriever_model,
    split="dev",
    pdf_base_dir=PDF_DIR,
    include_order_index=INCLUDE_ORDER_INDEX,
    use_pdf_extraction=USE_PDF_EXTRACTION,
    elements_by_poster=elements_by_poster,
)
test_dataset = PosterLayoutDataset(
    posters_df=posters_df,
    elements_df=elements_df,
    retriever_model=retriever_model,
    split="test",
    pdf_base_dir=PDF_DIR,
    include_order_index=INCLUDE_ORDER_INDEX,
    use_pdf_extraction=USE_PDF_EXTRACTION,
    elements_by_poster=elements_by_poster,
)
print(f"Datasets ready -> Train: {len(train_dataset):,} | Dev: {len(dev_dataset):,} | Test: {len(test_dataset):,}")

In [ ]:
# Structural, Anti-Leakage, Masking, and Element-ID Collation Verification
sample_indices = list(range(min(16, len(dev_dataset))))
samples = [dev_dataset[i] for i in sample_indices]

for i, s in enumerate(samples):
    t_pid = int(s["target_poster_id"].item())
    tmpl_pid = int(s["template_poster_id"].item())
    assert t_pid != tmpl_pid, f"Target poster {t_pid} retrieved itself!"
    assert tmpl_pid in train_pids, f"Retrieved template {tmpl_pid} is not from train split!"

    m = int(s["num_content"].item())
    assert s["content_feats"].shape[0] == m
    assert s["target_boxes"].shape == (m, 4)
    assert s["element_ids"].shape == (m,)
    assert (s["target_columns"] >= 0).all() and (s["target_columns"] < 5).all()

    tb = s["target_boxes"]
    assert (tb[:, 0] >= 0).all() and (tb[:, 1] >= 0).all()
    assert (tb[:, 2] > 0).all() and (tb[:, 3] > 0).all()
    assert (tb[:, 0] + tb[:, 2] <= 1.0 + 1e-5).all()
    assert (tb[:, 1] + tb[:, 3] <= 1.0 + 1e-5).all()

    # Verify missingness representation: when has_text_length == 0, log word/char == 0
    no_txt = s["content_feats"][:, 11] < 0.5
    assert torch.allclose(s["content_feats"][no_txt, 9:11], torch.zeros_like(s["content_feats"][no_txt, 9:11]))
    no_asp = s["content_feats"][:, 13] < 0.5
    assert torch.allclose(s["content_feats"][no_asp, 12], torch.ones_like(s["content_feats"][no_asp, 12]))

batch = collate_layout_batch(samples[:4])
for b_idx in range(4):
    m_len = int(samples[b_idx]["num_content"].item())
    k_len = int(samples[b_idx]["num_template"].item())
    assert batch["content_mask"][b_idx, :m_len].all() and (~batch["content_mask"][b_idx, m_len:]).all()
    assert batch["template_mask"][b_idx, :k_len].all() and (~batch["template_mask"][b_idx, k_len:]).all()
    assert torch.equal(batch["element_ids"][b_idx, :m_len], samples[b_idx]["element_ids"])
    assert (batch["element_ids"][b_idx, m_len:] == -1).all()

print("All Section B dataset & collation checks passed cleanly.")

## Section C — Dataset Inspection and Visual Diagnostics

In [ ]:
# Ground-truth distribution diagnostics across the training split
train_els = elements_df[elements_df["split"] == "train"]
train_pos = posters_df[(posters_df["split"] == "train") & (posters_df["total_element_count"] > 0)]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes[0, 0].hist(train_els["w_norm"], bins=40, alpha=0.65, label="w_norm")
axes[0, 0].hist(train_els["h_norm"], bins=40, alpha=0.65, label="h_norm")
axes[0, 0].set_title("Element Width & Height Distributions")
axes[0, 0].legend()

axes[0, 1].hist(train_pos["union_box_coverage"].dropna(), bins=35, color="#2ca02c", alpha=0.8)
axes[0, 1].set_title("Canvas Union Occupancy")

axes[0, 2].hist(train_pos["total_element_count"], bins=35, color="#9467bd", alpha=0.8)
axes[0, 2].set_title("Elements per Poster")

train_pos["estimated_column_count"].value_counts().sort_index().plot(kind="bar", ax=axes[1, 0], color="#1f77b4")
axes[1, 0].set_title("Estimated Column Count")

train_els["category_slug"].value_counts().plot(kind="bar", ax=axes[1, 1], color="#ff7f0e")
axes[1, 1].set_title("Content Category Distribution")

axes[1, 2].hist(train_pos["overlap_pair_fraction"].dropna(), bins=30, color="#d62728", alpha=0.8)
axes[1, 2].set_title("Ground-Truth Pairwise Overlap Fraction")

plt.tight_layout()
plt.show()

print("Ground-truth overlap_pair_fraction summary:")
print(train_pos["overlap_pair_fraction"].describe())

In [ ]:
# Inspect representative dev posters: content table + 4-panel visual comparison
adapter = LayoutAdapter(elements_df=elements_df, posters_df=posters_df)
baseline_model = None
if BASELINE_CKPT_PATH.exists():
    _ckpt = torch.load(BASELINE_CKPT_PATH, map_location="cpu", weights_only=False)
    _mcfg = extract_model_config_from_checkpoint(_ckpt)
    baseline_model = TemplateConditionedLayoutModel(
        content_dim=_mcfg["content_dim"],
        template_dim=_mcfg["template_dim"],
        d_model=_mcfg["d_model"],
        num_heads=_mcfg["num_heads"],
        encoder_layers=_mcfg["encoder_layers"],
        decoder_layers=_mcfg["decoder_layers"],
        ffn_dim=_mcfg["ffn_dim"],
        num_columns=_mcfg["num_columns"],
        dropout=_mcfg["dropout"],
        bbox_parameterization=_mcfg.get("bbox_parameterization", "independent"),
        use_anchor_conditioning=_mcfg.get("use_anchor_conditioning", False),
    )
    baseline_model.load_state_dict(_ckpt["model_state_dict"])
    baseline_model.eval()
    print(f"Loaded existing baseline checkpoint: {BASELINE_CKPT_PATH}")

def run_adapter_on_poster(pid: int):
    p_row = posters_df[posters_df["poster_id"] == pid].iloc[0]
    p_els = elements_by_poster[pid]
    req = simulate_request_from_heldout_poster(p_row, p_els)
    cands = retriever_model.retrieve_for_request(req, top_k=1, diversify=True, exclude_poster_id=pid)
    cand_row = posters_df[posters_df["poster_id"] == int(cands.iloc[0]["poster_id"])].iloc[0]
    res = adapter.adapt_single_candidate(req, cand_row, candidate_rank=1)
    boxes = np.array([s.target_bbox_norm for s in res.optimization_result.adapted_slots], dtype=np.float32)
    cats = [s.content_type for s in res.optimization_result.adapted_slots]
    return boxes, cats, res

for idx in [0, 1]:
    row = dev_dataset.split_posters.iloc[idx]
    raw_sample = build_synthetic_request_and_sample(
        row, elements_df, retriever_model, pdf_base_dir=PDF_DIR,
        elements_by_poster=elements_by_poster, use_pdf_extraction=USE_PDF_EXTRACTION,
    )
    display_df = pd.DataFrame([
        {
            "element_id": eid,
            "category": item.category_slug,
            "has_text": item.has_text_length,
            "words": int(item.word_count) if item.has_text_length else None,
            "has_aspect": item.has_aspect_ratio,
            "aspect_ratio": round(item.aspect_ratio, 2) if item.has_aspect_ratio else None,
        }
        for eid, item in zip(raw_sample.element_ids, raw_sample.content_items)
    ])
    print(f"\n=== Dev Sample {idx} (PID={raw_sample.target_poster_id}, Template={raw_sample.template_poster_id}) ===")
    display(display_df.head(12))

    item_t = dev_dataset[idx]
    neural_boxes = np.zeros((0, 4))
    neural_orders = []
    if baseline_model is not None:
        with torch.no_grad():
            c_in = item_t["content_feats"].unsqueeze(0)
            t_in = item_t["template_feats"].unsqueeze(0)
            out = baseline_model(c_in, t_in, torch.ones(1, c_in.shape[1], dtype=torch.bool), torch.ones(1, t_in.shape[1], dtype=torch.bool))
            neural_boxes = out["pred_boxes"][0].numpy()
            neural_orders = np.argsort(np.argsort(out["pred_order_scores"][0].numpy())).tolist()

    ad_boxes, ad_cats, _ = run_adapter_on_poster(raw_sample.target_poster_id)
    cats = [KNOWN_CATEGORIES[int(i)] for i in torch.argmax(item_t["content_feats"][:, :9], dim=-1)]
    tcats = [KNOWN_CATEGORIES[int(i)] for i in torch.argmax(item_t["template_feats"][:, :9], dim=-1)]
    fig = render_four_panel_comparison(
        target_boxes=item_t["target_boxes"].numpy(),
        target_categories=cats,
        target_orders=item_t["target_orders"].numpy().tolist(),
        template_boxes=item_t["template_feats"][:, 9:13].numpy(),
        template_categories=tcats,
        neural_boxes=neural_boxes,
        neural_categories=cats,
        neural_orders=neural_orders,
        adapter_boxes=ad_boxes,
        adapter_categories=ad_cats,
        poster_id=raw_sample.target_poster_id,
        template_id=raw_sample.template_poster_id,
        title_suffix="(Existing Checkpoint)" if baseline_model is not None else "(None)",
    )
    plt.show()

## Section D — Pre-Training Baselines & Input Representation Ablation

Evaluates `LayoutAdapter + HiGHS`, the existing checkpoint (raw predictions without LP repair), and tests permutation equivariance (`include_order_index=True` vs `include_order_index=False`).

In [ ]:
BASELINE_EVAL_N = min(100, len(dev_dataset))
evaluator = LayoutEvaluator()

# 1. LayoutAdapter + HiGHS baseline on dev subset
adapter_metrics_rows = []
for i in range(BASELINE_EVAL_N):
    pid = int(dev_dataset.split_posters.iloc[i]["poster_id"])
    _, _, res = run_adapter_on_poster(pid)
    adapter_metrics_rows.append(res.to_summary_dict())
adapter_baseline_df = pd.DataFrame(adapter_metrics_rows)
print("=== LayoutAdapter + HiGHS Baseline Summary (Dev Subset) ===")
print(adapter_baseline_df[["is_feasible", "composite_quality_score", "overflow_rate", "overlap_area", "aspect_distortion", "whitespace_utilization"]].mean())

# 2. Existing checkpoint raw neural baseline (NO LP repair)
if baseline_model is not None:
    from torch.utils.data import DataLoader, Subset
    sub_loader = DataLoader(Subset(dev_dataset, range(BASELINE_EVAL_N)), batch_size=16, shuffle=False, collate_fn=collate_layout_batch)
    acc = {}
    nb = 0
    with torch.no_grad():
        for b in sub_loader:
            preds = baseline_model(b["content_feats"], b["template_feats"], b["content_mask"], b["template_mask"])
            m = evaluator.evaluate_batch(preds, b)
            for k, v in m.items():
                acc[k] = acc.get(k, 0.0) + v
            nb += 1
    print("\n=== Existing Checkpoint Raw Neural Metrics (No Repair) ===")
    print({k: round(v / max(1, nb), 4) for k, v in acc.items()})

# 3. Permutation-equivariance check: with vs without sequence-index feature (dim 17)
if RUN_ORDER_INDEX_ABLATION:
    test_item = dev_dataset[0]
    c0 = test_item["content_feats"].unsqueeze(0).clone()
    t0 = test_item["template_feats"].unsqueeze(0)
    cm0 = torch.ones(1, c0.shape[1], dtype=torch.bool)
    tm0 = torch.ones(1, t0.shape[1], dtype=torch.bool)
    perm = torch.randperm(c0.shape[1], generator=torch.Generator().manual_seed(SEED))

    probe_model = TemplateConditionedLayoutModel(
        d_model=TRAIN_CONFIG.d_model,
        num_heads=TRAIN_CONFIG.num_heads,
        encoder_layers=TRAIN_CONFIG.encoder_layers,
        decoder_layers=TRAIN_CONFIG.decoder_layers,
        ffn_dim=TRAIN_CONFIG.ffn_dim,
        bbox_parameterization=TRAIN_CONFIG.bbox_parameterization,
        use_anchor_conditioning=TRAIN_CONFIG.use_anchor_conditioning,
    ).eval()
    with torch.no_grad():
        # With arbitrary dim 17 populated
        c_with_idx = c0.clone()
        c_with_idx[0, :, 17] = torch.linspace(0.0, 0.9, c0.shape[1])
        p_a = probe_model(c_with_idx, t0, cm0, tm0)["pred_boxes"][0]
        p_b = probe_model(c_with_idx[:, perm], t0, cm0, tm0)["pred_boxes"][0]
        # Without sequence index (dim 17 = 0.0)
        c_no_idx = c0.clone()
        c_no_idx[0, :, 17] = 0.0
        q_a = probe_model(c_no_idx, t0, cm0, tm0)["pred_boxes"][0]
        q_b = probe_model(c_no_idx[:, perm], t0, cm0, tm0)["pred_boxes"][0]

    print("\n=== Permutation Equivariance Check (Max |f(P*X) - P*f(X)|) ===")
    print(f"With dim 17 sequence index    : {(p_a[perm] - p_b).abs().max().item():.6f} (breaks equivariance if dim 17 shuffled)")
    print(f"Without dim 17 sequence index : {(q_a[perm] - q_b).abs().max().item():.6e} (exact equivariance)")

## Section E — Model Initialization and Diagnostics

Initializes the anchor-conditioned bounded `TemplateConditionedLayoutModel`, inspects parameter counts, gradient flow, initial prediction distributions, template sensitivity, and runs a fixed tiny-batch overfitting diagnostic.

In [ ]:
model = TemplateConditionedLayoutModel(
    content_dim=18,
    template_dim=16,
    d_model=TRAIN_CONFIG.d_model,
    num_heads=TRAIN_CONFIG.num_heads,
    encoder_layers=TRAIN_CONFIG.encoder_layers,
    decoder_layers=TRAIN_CONFIG.decoder_layers,
    ffn_dim=TRAIN_CONFIG.ffn_dim,
    num_columns=5,
    dropout=TRAIN_CONFIG.dropout,
    bbox_parameterization=TRAIN_CONFIG.bbox_parameterization,
    use_anchor_conditioning=TRAIN_CONFIG.use_anchor_conditioning,
    min_box_width=TRAIN_CONFIG.min_box_width,
    min_box_height=TRAIN_CONFIG.min_box_height,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal params: {total_params:,} | Trainable params: {trainable_params:,} | Device: {DEVICE}")
print(f"Loss weights: {TRAIN_CONFIG.loss_weights}")

# Single-batch forward/backward sanity check
sanity_batch = collate_layout_batch([train_dataset[i] for i in range(4)])
sanity_dev = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in sanity_batch.items()}
loss_fn_sanity = MultiTaskLayoutLoss(weights=TRAIN_CONFIG.loss_weights)

model.train()
sanity_preds = model(sanity_dev["content_feats"], sanity_dev["template_feats"], sanity_dev["content_mask"], sanity_dev["template_mask"])
sanity_loss, sanity_ld = loss_fn_sanity(sanity_preds, sanity_dev)
sanity_loss.backward()

grad_norms = [float(p.grad.detach().norm().item()) for p in model.parameters() if p.grad is not None]
has_nan_grad = any(torch.isnan(p.grad).any().item() or torch.isinf(p.grad).any().item() for p in model.parameters() if p.grad is not None)
model.zero_grad(set_to_none=True)

with torch.no_grad():
    model.eval()
    p_valid = sanity_preds["pred_boxes"][sanity_dev["content_mask"]].detach().cpu()
    p_zero_t = model(sanity_dev["content_feats"], torch.zeros_like(sanity_dev["template_feats"]), sanity_dev["content_mask"], sanity_dev["template_mask"])["pred_boxes"][sanity_dev["content_mask"]].detach().cpu()
    init_eval = evaluator.evaluate_batch(sanity_preds, sanity_dev)

print(f"Sanity loss_dict        : {sanity_ld}")
print(f"Grad norms min/mean/max : {min(grad_norms):.4e} / {np.mean(grad_norms):.4e} / {max(grad_norms):.4e} | NaN/Inf: {has_nan_grad}")
print(f"Template sensitivity    : {(p_valid - p_zero_t).abs().max().item():.4f}")
print(f"Init center std (x, y)  : ({init_eval['pred_center_std_x']:.4f}, {init_eval['pred_center_std_y']:.4f})")
print(f"Init wh std (w, h)      : ({init_eval['pred_wh_std_w']:.4f}, {init_eval['pred_wh_std_h']:.4f})")
print(f"Init overlap rate       : {init_eval['unintended_overlap_rate']:.4f}")

In [ ]:
# Fixed Tiny-Batch Overfitting Diagnostic (disposable clone; does not mutate main `model`)
import copy

overfit_model = copy.deepcopy(model).to(DEVICE)
overfit_opt = torch.optim.AdamW(overfit_model.parameters(), lr=1e-3, weight_decay=0.0)
overfit_loss_fn = MultiTaskLayoutLoss(weights=TRAIN_CONFIG.loss_weights)
overfit_history = []

overfit_model.train()
for step in range(1, 61):
    overfit_opt.zero_grad(set_to_none=True)
    preds_ov = overfit_model(
        sanity_dev["content_feats"],
        sanity_dev["template_feats"],
        sanity_dev["content_mask"],
        sanity_dev["template_mask"],
    )
    loss_ov, ld_ov = overfit_loss_fn(preds_ov, sanity_dev)
    loss_ov.backward()
    torch.nn.utils.clip_grad_norm_(overfit_model.parameters(), 1.0)
    overfit_opt.step()

    if step in (1, 15, 30, 60):
        ev_ov = evaluator.evaluate_batch(preds_ov, sanity_dev)
        overfit_history.append({
            "step": step,
            "loss_total": round(ld_ov["loss_total"], 4),
            "loss_box": round(ld_ov["loss_box"], 4),
            "loss_overlap": round(ld_ov["loss_overlap"], 4),
            "iou_mean": round(ev_ov["iou_mean"], 4),
            "overlap_rate": round(ev_ov["unintended_overlap_rate"], 4),
            "center_std_x": round(ev_ov["pred_center_std_x"], 4),
        })

display(pd.DataFrame(overfit_history))
del overfit_model, overfit_opt

## Section F — Full Training (Complete Train Split + Per-Epoch Dev Evaluation)

Trains `TemplateConditionedLayoutModel` across all training posters with AMP, cosine warmup, early stopping, checkpointing, and ClearML tracking.

In [ ]:
trainer = LayoutTrainer(
    model=model,
    config=TRAIN_CONFIG,
    train_dataset=train_dataset,
    val_dataset=dev_dataset,
)

if RESUME_FROM_CHECKPOINT is not None and Path(RESUME_FROM_CHECKPOINT).exists():
    print(f"Resuming from checkpoint: {RESUME_FROM_CHECKPOINT}")
    trainer.load_checkpoint(Path(RESUME_FROM_CHECKPOINT))

history = trainer.train()

# Plot main training curves and every individual loss component
fig_curves = render_training_curves(history, save_path=OUTPUT_DIR / "training_curves.png")
plt.show()

loss_names = ["loss_box", "loss_size", "loss_giou", "loss_bound", "loss_overlap", "loss_col", "loss_aspect", "loss_order", "loss_macro"]
fig_losses, axes = plt.subplots(3, 3, figsize=(16, 11))
epochs_x = history["epoch"]
for ax, lname in zip(axes.ravel(), loss_names):
    tr_key = f"train_{lname}"
    val_key = f"val_{lname}"
    if tr_key in history:
        ax.plot(epochs_x, history[tr_key], label="train")
    if val_key in history:
        ax.plot(epochs_x, history[val_key], label="val")
    ax.set_title(lname)
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "individual_loss_components.png", dpi=160, bbox_inches="tight")
plt.show()

## Section G — Comprehensive Dev Evaluation (+ Post-Selection Held-Out Test Evaluation)

In [ ]:
from torch.utils.data import DataLoader

best_ckpt_path = CHECKPOINT_DIR / "best_model.pt"
best_state = torch.load(best_ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_state["model_state_dict"])
model.eval()
print(f"Loaded best checkpoint from epoch {best_state['epoch'] + 1}: {best_ckpt_path}")

def evaluate_full_split(eval_model, dataset, split_name: str):
    loader = DataLoader(dataset, batch_size=TRAIN_CONFIG.batch_size, shuffle=False, collate_fn=collate_layout_batch)
    agg_metrics = {}
    num_b = 0
    poster_rows = []
    element_rows = []
    with torch.no_grad():
        for b in loader:
            b_dev = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in b.items()}
            preds = eval_model(b_dev["content_feats"], b_dev["template_feats"], b_dev["content_mask"], b_dev["template_mask"])
            bm = evaluator.evaluate_batch(preds, b_dev)
            for k, v in bm.items():
                agg_metrics[k] = agg_metrics.get(k, 0.0) + v
            num_b += 1

            pb = preds["pred_boxes"].detach().cpu()
            tb = b["target_boxes"]
            cm = b["content_mask"]
            cf = b["content_feats"]
            for bi in range(pb.shape[0]):
                m_cnt = int(cm[bi].sum().item())
                p_sub = pb[bi, :m_cnt]
                t_sub = tb[bi, :m_cnt]
                # Per-element IoU & errors for category breakdown
                ix0 = torch.max(p_sub[:, 0], t_sub[:, 0])
                iy0 = torch.max(p_sub[:, 1], t_sub[:, 1])
                ix1 = torch.min(p_sub[:, 0] + p_sub[:, 2], t_sub[:, 0] + t_sub[:, 2])
                iy1 = torch.min(p_sub[:, 1] + p_sub[:, 3], t_sub[:, 1] + t_sub[:, 3])
                inter = torch.clamp(ix1 - ix0, min=0.0) * torch.clamp(iy1 - iy0, min=0.0)
                union = p_sub[:, 2] * p_sub[:, 3] + t_sub[:, 2] * t_sub[:, 3] - inter
                ious = (inter / torch.clamp(union, min=1e-6)).numpy()
                c_idx = torch.argmax(cf[bi, :m_cnt, :9], dim=-1).numpy()
                poster_rows.append({
                    "poster_id": int(b["target_poster_ids"][bi].item()),
                    "num_elements": m_cnt,
                    "complexity_bucket": "<=15" if m_cnt <= 15 else ("16-30" if m_cnt <= 30 else ">30"),
                    "mean_iou": float(np.mean(ious)),
                })
                for ei in range(m_cnt):
                    element_rows.append({
                        "category": KNOWN_CATEGORIES[int(c_idx[ei])],
                        "iou": float(ious[ei]),
                        "wh_l1": float(0.5 * (abs(p_sub[ei, 2] - t_sub[ei, 2]) + abs(p_sub[ei, 3] - t_sub[ei, 3]))),
                    })

    summary = {k: v / max(1, num_b) for k, v in agg_metrics.items()}
    print(f"\n=== {split_name.upper()} EVALUATION (N={len(dataset):,}) ===")
    for grp, keys in {
        "Geometry": ["iou_mean", "giou_mean", "center_l1_error", "wh_l1_error", "canvas_violation_rate"],
        "Collisions": ["unintended_overlap_rate", "pairwise_iou_mean", "unintended_overlap_mean_area"],
        "Structure": ["column_accuracy", "macro_column_accuracy", "reading_order_kendall_tau", "aspect_ratio_log_error"],
        "Diversity & Collapse": ["pred_center_std_x", "pred_center_std_y", "pred_wh_std_w", "pred_wh_std_h", "near_identical_box_rate"],
    }.items():
        print(f"[{grp}]", {k: round(summary.get(k, 0.0), 4) for k in keys})
    return summary, pd.DataFrame(poster_rows), pd.DataFrame(element_rows)

dev_summary, dev_poster_df, dev_el_df = evaluate_full_split(model, dev_dataset, "dev")
print("\nDev IoU by Complexity:")
display(dev_poster_df.groupby("complexity_bucket")["mean_iou"].agg(["count", "mean"]))
print("\nDev Metrics by Content Category:")
display(dev_el_df.groupby("category")[["iou", "wh_l1"]].agg(["count", "mean"]))

In [ ]:
# Final Held-Out Test Split Evaluation (run ONLY after model selection is finalized)
RUN_FINAL_TEST_EVAL = False
test_summary = {}
if RUN_FINAL_TEST_EVAL:
    test_summary, test_poster_df, test_el_df = evaluate_full_split(model, test_dataset, "test")
else:
    print("Held-out test evaluation skipped (set RUN_FINAL_TEST_EVAL = True once model selection is complete).")

## Section H — Visual Results (4-Panel Comparisons)

Generates 4-panel comparisons (`Ground Truth`, `Retrieved Template`, `Raw Neural Prediction` without LP repair, and `LayoutAdapter + HiGHS`) across diverse dev posters.

In [ ]:
# Select diverse dev posters spanning portrait/landscape, low/high counts, and columns
dev_meta = dev_dataset.split_posters.copy()
selected_indices = []
for cond in [
    (dev_meta["orientation"] == "portrait") & (dev_meta["total_element_count"] <= 18),
    (dev_meta["orientation"] == "portrait") & (dev_meta["total_element_count"] >= 28),
    (dev_meta["orientation"] == "landscape") & (dev_meta["estimated_column_count"] >= 3),
    (dev_meta["orientation"] == "landscape") & (dev_meta["count_figure"] >= 3),
]:
    matches = dev_meta.index[cond].tolist()
    for m_idx in matches:
        if m_idx not in selected_indices:
            selected_indices.append(m_idx)
            break
if not selected_indices:
    selected_indices = list(range(min(4, len(dev_dataset))))

model.eval()
for rank_i, ds_idx in enumerate(selected_indices, start=1):
    item_t = dev_dataset[ds_idx]
    pid = int(item_t["target_poster_id"].item())
    tmpl_pid = int(item_t["template_poster_id"].item())
    with torch.no_grad():
        c_in = item_t["content_feats"].unsqueeze(0).to(DEVICE)
        t_in = item_t["template_feats"].unsqueeze(0).to(DEVICE)
        cm = torch.ones(1, c_in.shape[1], dtype=torch.bool, device=DEVICE)
        tm = torch.ones(1, t_in.shape[1], dtype=torch.bool, device=DEVICE)
        out = model(c_in, t_in, cm, tm)
        raw_boxes = out["pred_boxes"][0].detach().cpu().numpy()
        raw_orders = np.argsort(np.argsort(out["pred_order_scores"][0].detach().cpu().numpy())).tolist()

    cats = [KNOWN_CATEGORIES[int(i)] for i in torch.argmax(item_t["content_feats"][:, :9], dim=-1)]
    tcats = [KNOWN_CATEGORIES[int(i)] for i in torch.argmax(item_t["template_feats"][:, :9], dim=-1)]
    ad_boxes, ad_cats, _ = run_adapter_on_poster(pid)
    fig = render_four_panel_comparison(
        target_boxes=item_t["target_boxes"].numpy(),
        target_categories=cats,
        target_orders=item_t["target_orders"].numpy().tolist(),
        template_boxes=item_t["template_feats"][:, 9:13].numpy(),
        template_categories=tcats,
        neural_boxes=raw_boxes,
        neural_categories=cats,
        neural_orders=raw_orders,
        adapter_boxes=ad_boxes,
        adapter_categories=ad_cats,
        poster_id=pid,
        template_id=tmpl_pid,
        save_path=OUTPUT_DIR / f"dev_4panel_{rank_i}_pid_{pid}.png",
        title_suffix="(Raw Output)",
    )
    plt.show()

## Section I — Checkpoint Persistence & Manual ClearML Model Registration

Saves evaluation metrics locally and provides an explicitly guarded ClearML registration cell. **Do not overwrite `0.1.0`** and **do not auto-promote to production**.

In [ ]:
eval_payload = {
    "checkpoint_path": str(best_ckpt_path),
    "model_config": extract_model_config_from_checkpoint(best_state),
    "training_config": extract_training_config_from_checkpoint(best_state),
    "dev_summary": dev_summary,
    "test_summary": test_summary,
}
(OUTPUT_DIR / "eval_metrics.json").write_text(json.dumps(eval_payload, indent=2), encoding="utf-8")
print(f"Saved evaluation summary to {OUTPUT_DIR / 'eval_metrics.json'}")

# Explicit manual registration guard
REGISTER_CANDIDATE = False
NEW_SEMANTIC_VERSION = "0.2.0"  # Must be supplied manually; never overwrite 0.1.0
CLEARML_TRAINING_TASK_ID = best_state.get("clearml_task_id") or (trainer.clearml_task.id if trainer.clearml_task else "")

if REGISTER_CANDIDATE:
    if NEW_SEMANTIC_VERSION.strip().lstrip("v") == "0.1.0":
        raise ValueError("Refusing to overwrite poster_layout_generator-v0.1.0. Supply a new version.")
    if not CLEARML_TRAINING_TASK_ID:
        raise ValueError("Missing CLEARML_TRAINING_TASK_ID for provenance verification.")
    model_id = register_model(
        checkpoint_path=best_ckpt_path,
        training_task_id=CLEARML_TRAINING_TASK_ID,
        semantic_version=NEW_SEMANTIC_VERSION,
        dataset_identifier=TRAIN_CONFIG.dataset_version,
        validation_metrics=dev_summary,
        lifecycle_status="candidate",
        comment="Anchor-conditioned bounded layout generator trained on pooled 70/15/15 split.",
        verify_provenance=True,
    )
    print(f"Registered ClearML candidate model ID: {model_id}")
else:
    print("Registration skipped (set REGISTER_CANDIDATE = True to register manually as 'candidate').")